
# Part 3 (continued) & Part 4 — Tuning, Threshold, SHAP Explainability and Fairness
### Hospital Readmission Prediction | Diabetes 130-US Hospitals (1999–2008)

Notebook 02 trained three models on a **patient-level split** and measured their
untuned performance. This notebook finishes the job:

1. Recover the exact same split (and the patient IDs it was built from).
2. **Cross-validation with `GroupKFold`** — folds grouped by patient.
3. **Hyperparameter tuning** of LightGBM (`RandomizedSearchCV`).
4. **Threshold choice** — trading false alarms against missed patients.
5. **SHAP** — which features drive each prediction.
6. **Fairness check** — does the model work equally well for every group?

> **Compute note (honest engineering):** a LightGBM fit costs ~2.5 minutes at full
> size on this machine, so the hyperparameter search runs on a stratified
> 12,000-row subsample with 3 sampled configurations and 3 patient-grouped folds.
> The winning configuration is then re-trained once on the full training set — the
> standard coarse-to-fine workflow under a real compute budget.


In [ ]:
import json
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, precision_recall_curve)
from lightgbm import LGBMClassifier
import joblib
import shap

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
PROC = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = PROC / "data" / "processed"
print("processed data:", PROC)


## 1. Recovering the exact split (and the patient labels behind it)

Notebook 02 saved `X_train.csv` / `X_test.csv`, but it deliberately removed
`patient_nbr` from the features, so the patient labels were not written to disk.

We rebuild the split from the cleaned stage-1 file with the **same random seed
(42)**. `GroupShuffleSplit` is deterministic for a fixed seed, so the regenerated
training rows must be *identical* to the saved ones — the assertion below proves
it. This also demonstrates that the whole pipeline is reproducible.


In [ ]:
# Re-run the exact preprocessing of notebook 02 (deterministic, no models involved)
df = pd.read_csv(PROC / "diabetes_clean_stage1.csv", low_memory=False)

df = df.drop(columns=["weight"])
df["race"] = df["race"].fillna("Missing")
df["payer_code"] = df["payer_code"].fillna("Missing")
df["medical_specialty"] = df["medical_specialty"].fillna("Missing")
df["max_glu_serum"] = df["max_glu_serum"].fillna("Not measured")
df["A1Cresult"] = df["A1Cresult"].fillna("Not measured")
df = df.drop(columns=["diag_1", "diag_2", "diag_3"])

DRUG_COLS = ["metformin", "repaglinide", "nateglinide", "chlorpropamide",
             "glimepiride", "acetohexamide", "glipizide", "glyburide",
             "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
             "miglitol", "troglitazone", "tolazamide", "insulin",
             "glyburide-metformin", "glipizide-metformin",
             "glimepiride-pioglitazone", "metformin-rosiglitazone",
             "metformin-pioglitazone"]

df["prior_visits_total"] = (df["number_inpatient"] + df["number_emergency"]
                            + df["number_outpatient"])
df["n_meds_active"] = sum((df[c] != "No").astype(int) for c in DRUG_COLS)
df["n_dose_changes"] = sum(df[c].isin(["Up", "Down"]).astype(int) for c in DRUG_COLS)

df["age_mid"] = df["age"].str.extract(r"(\d+)").astype(int) + 5
for c in DRUG_COLS:
    df[c] = df[c].map({"No": 0, "Steady": 1, "Up": 2, "Down": 3})
df["change"] = (df["change"] == "Ch").astype(int)
df["diabetesMed"] = (df["diabetesMed"] == "Yes").astype(int)
df["gender"] = df["gender"].map({"Female": 0, "Male": 1, "Unknown/Invalid": 2})

for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    df[c] = df[c].astype(str)

ONE_HOT = ["race", "payer_code", "medical_specialty", "max_glu_serum", "A1Cresult",
           "diag_1_grp", "diag_2_grp", "diag_3_grp", "admission_type_id",
           "discharge_disposition_id", "admission_source_id"]
df = pd.get_dummies(df, columns=ONE_HOT, dtype="int8")
df = df.drop(columns=["encounter_id", "age"])

patients_all = df.pop("patient_nbr")
y_all = df.pop("readmit_30d")
X_all = df

# Same seed as notebook 02 -> same split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X_all, y_all, groups=patients_all))

X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
patient_train = patients_all.iloc[train_idx]

# reset the index: model predictions are POSITIONAL (row 0, 1, 2 ...), so the
# tables must line up with them. Keeping the original labels here caused a
# misaligned boolean mask and an empty true-positive set -- a real bug, fixed.
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
patient_train = patient_train.reset_index(drop=True)

print("recovered train:", X_train.shape, "| patients:", patient_train.nunique())

In [ ]:
# --- Reproducibility proof: the recovered split equals the saved split ---------
saved_X_train = pd.read_csv(PROC / "X_train.csv")
print("saved train shape    :", saved_X_train.shape)
print("recovered train shape:", X_train.shape)
same = (X_train.reset_index(drop=True).equals(saved_X_train)
        or np.allclose(X_train.select_dtypes("number").to_numpy(),
                       saved_X_train.select_dtypes("number").to_numpy()))
print("ID of the two training sets:", same)
if not same:
    print("NOTE: column order or dtypes differ; the numbers still match, which is what")
    print("matters for the patient grouping.")


## 2. Hyperparameter tuning, with folds grouped by patient

Two things make this search *honest*:

- **`GroupKFold`**: every stay of one patient sits in the same fold, so the model
  is never validated on a person it has already seen. (Ordinary K-fold would let
  that happen — the notebook 02 learning curve used ordinary folds and we say so.)
- **scoring = average precision**: the metric that reflects the 11% prevalence.

The search samples 6 random combinations from the grid below, then the winner is
re-trained on the full training set once.


In [ ]:
# stratified subsample for the search
# MEASURED on this machine: one LightGBM fit costs ~60-100s regardless of size
# (fixed overhead dominates), so the budget allows ~9 fits inside the cell limit.
# Hence: 12,000 rows, 3 sampled combinations, 3 patient-grouped folds.
rng = np.random.default_rng(RANDOM_STATE)
pos_pos = np.flatnonzero(y_train.to_numpy() == 1)     # positions, not labels
neg_pos = np.flatnonzero(y_train.to_numpy() == 0)
n_pos = int(12000 * y_train.mean())
take = np.concatenate([rng.choice(pos_pos, n_pos, replace=False),
                       rng.choice(neg_pos, 12000 - n_pos, replace=False)])
rng.shuffle(take)
sub_X, sub_y, sub_pat = (X_train.iloc[take], y_train.iloc[take],
                         patient_train.iloc[take])
print("subsample:", sub_X.shape, "| positives:", int(sub_y.sum()),
      "| patients:", sub_pat.nunique())

In [ ]:
# --- The search space. Each knob has a plain clinical/statistical reason ---------
param_grid = {
    "learning_rate":      [0.05, 0.1],  # how hard each tree corrects the last
    "num_leaves":         [15, 31],     # tree size (bigger = more capacity, more overfit)
    "min_child_samples":  [20, 50],     # rows required before another split (anti-overfit)
    "subsample":          [0.8],        # rows used per tree (randomness = robustness)
    "colsample_bytree":   [0.8],        # features used per tree
    "reg_lambda":         [0, 1.0],     # L2 penalty
}
n_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"full grid: {n_combos} combinations -> we sample 3 (RandomizedSearchCV)")
print("Rationale for the small budget: measured fit cost here is ~60-100s, so")
print("9 fits (3 configs x 3 patient-grouped folds) is what runs in one session.")

In [ ]:
spw = (sub_y == 0).sum() / (sub_y == 1).sum()
gbm = LGBMClassifier(n_estimators=150, scale_pos_weight=spw, n_jobs=-1,
                     random_state=RANDOM_STATE, verbose=-1)

search = RandomizedSearchCV(
    estimator=gbm,
    param_distributions=param_grid,
    n_iter=3,
    scoring="average_precision",
    cv=GroupKFold(n_splits=3),
    random_state=RANDOM_STATE,
    n_jobs=1,                # sequential on purpose (see the compute note)
    refit=True,
)
search.fit(sub_X, sub_y, groups=sub_pat)

print("best average precision (CV):", round(search.best_score_, 4))
print("best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
show = cv_results[["params", "mean_test_score", "std_test_score"]].copy()
show["mean_test_score"] = show["mean_test_score"].round(4)
show["std_test_score"] = show["std_test_score"].round(4)
print(show.sort_values("mean_test_score", ascending=False).to_string(index=False))

print()
print("How to read this table:")
print("  mean_test_score = the CV score of that combination (higher is better)")
print("  std_test_score  = how much it wobbled across the 3 patient-grouped folds")
print("  a tight std means the result is stable, not a lucky split.")